# 1. Load dữ liệu 

In [6]:
import pandas as pd
df = pd.read_csv("/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")
print("5 dòng đầu tiên: ", df)

5 dòng đầu tiên:                                                    review sentiment
0      One of the other reviewers has mentioned that ...  positive
1      A wonderful little production. <br /><br />The...  positive
2      I thought this was a wonderful way to spend ti...  positive
3      Basically there's a family where a little boy ...  negative
4      Petter Mattei's "Love in the Time of Money" is...  positive
...                                                  ...       ...
49995  I thought this movie did a down right good job...  positive
49996  Bad plot, bad dialogue, bad acting, idiotic di...  negative
49997  I am a Catholic taught in parochial elementary...  negative
49998  I'm going to have to disagree with the previou...  negative
49999  No one expects the Star Trek movies to be high...  negative

[50000 rows x 2 columns]


# 2. Làm sạch dữ liệu
- mục tiêu là loại bỏ các thẻ html dư thừa, url
- xóa những khoảng trắng dữ thừa
- xóa những ký tự đặc biệt

In [7]:
import re
from bs4 import BeautifulSoup 
# r: là regrex (biểu thức chính quy)
# nó giúp /n => xuống dòng
# nếu không có r thì nó vẫn hiểu /n

# \S: là ký tự khác khoảng trắng
# \s: là khoảng trắng
# \n: xuống hàng
# a-z: là a,b,c,d,...z
# A-Z: là A,B,C,D,...Z
# ^: là ngược với những cái còn lại

def clean_text(text):
    text = BeautifulSoup(text, "html.parser").get_text()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"[^a-zA-Z\s]", "", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

df["review"] = df["review"].apply(clean_text)
print(df)



                                                  review sentiment
0      One of the other reviewers has mentioned that ...  positive
1      A wonderful little production The filming tech...  positive
2      I thought this was a wonderful way to spend ti...  positive
3      Basically theres a family where a little boy J...  negative
4      Petter Matteis Love in the Time of Money is a ...  positive
...                                                  ...       ...
49995  I thought this movie did a down right good job...  positive
49996  Bad plot bad dialogue bad acting idiotic direc...  negative
49997  I am a Catholic taught in parochial elementary...  negative
49998  Im going to have to disagree with the previous...  negative
49999  No one expects the Star Trek movies to be high...  negative

[50000 rows x 2 columns]


# 3. Chuẩn hóa text
- mục tiêu: đưa text về 1 dạng duy nhất
- lower case
- xóa những khoảng trắng trùng lặp
- tokenize: vì máy tính không thể hiểu được văn bản, do đó phải ánh xạ từ văn bản thành các con số
- link tham khảo tokenize: https://www.youtube.com/watch?v=hL4ZnAWSyuU

- ví dụ:
Câu đầu vào: "I love coding"

Bước 1 (Tách): ['I', 'love', 'coding']

Bước 2 (Tra từ điển): I có mã là 101, love có mã là 423, coding có mã là 1256.

Kết quả máy tính nhận được: [101, 423, 1256]

## 3 cấp độ tách từ
#### tách theo từ:
- "Tôi đi học" -> ["Tôi", "đi", "học"], nhược điểm: nếu gặp từ mới hay từ viết tắt trong câu thì hệ thống không biết

#### Tách theo ký tự
- "Cat" -> ["C", "a", "t"], nhược điểm: dài, tốn bộ nhớ, làm mất ý nghĩa của từ

#### Tách theo từ con:
- các từ phổ biến thì giữ nguyên, còn các từ còn lại thì bị bẻ đôi
-  Ví dụ: Từ "unhappiness" (sự không hạnh phúc) có thể bị tách thành: ['un', 'happi', 'ness'].

Ưu điểm: Vừa giữ được ý nghĩa của từ, từ điển không quá lớn, lại vừa xử lý được các từ mới bằng cách ghép các mảnh nhỏ lại với nhau.

In [8]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text

df["review"] = df["review"].apply(normalize_text)

In [9]:
print(df.head())

                                              review sentiment
0  one of the other reviewers has mentioned that ...  positive
1  a wonderful little production the filming tech...  positive
2  i thought this was a wonderful way to spend ti...  positive
3  basically theres a family where a little boy j...  negative
4  petter matteis love in the time of money is a ...  positive


# 4. Xóa trùng lặp(Deduplicate)

In [10]:
before = len(df)

df = df.drop_duplicates(subset=["review"])

after = len(df)
print("Trước khi xóa trùng lặp: ", before)
print("Sau khi xóa trùng lặp: ", after)

Trước khi xóa trùng lặp:  50000
Sau khi xóa trùng lặp:  49580


# 5. Label / annotation
- Label (Nhãn): Là đáp án cuối cùng. Bạn chỉ gắn một cái tên hoặc một danh mục cho toàn bộ dữ liệu.

Ví dụ: Nhìn bức ảnh và nói: "Đây là con chó". Chữ "con chó" là Label.

- Annotation (Chú thích dữ liệu): Là quá trình làm chi tiết dữ liệu. Bạn không chỉ gọi tên mà còn chỉ rõ vị trí, hình dáng hoặc cấu trúc của nó.

Ví dụ: Vẽ một khung hình chữ nhật bao quanh con chó trong ảnh để máy biết nó nằm ở tọa độ nào. Hành động vẽ và định vị đó là Annotation.

In [11]:
label_map = {
    "positive": 1,
    "negative": 0
}
df["sentiment"] = df["sentiment"].map(label_map)
print(df.head())

                                              review  sentiment
0  one of the other reviewers has mentioned that ...          1
1  a wonderful little production the filming tech...          1
2  i thought this was a wonderful way to spend ti...          1
3  basically theres a family where a little boy j...          0
4  petter matteis love in the time of money is a ...          1


# 6. QA/QC
- kiểm tra dữ liệu lỗi

In [12]:
print(df.isnull().sum())

review       0
sentiment    0
dtype: int64


In [13]:
print(df.value_counts())

review                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [14]:
print(df["review"].str.len() > 10)

0        True
1        True
2        True
3        True
4        True
         ... 
49995    True
49996    True
49997    True
49998    True
49999    True
Name: review, Length: 49580, dtype: bool


In [15]:
print(df.sample(5))

                                                  review  sentiment
45483  i loved that the mood was light and airy i lov...          1
33128  twelve monkeys is odd and disturbing yet being...          1
28854  i just saw checking out at the philadelphia fi...          1
21853  spoiler spoilers spoilers i found the film amu...          1
21617  chris ricci sleepwalks her way through most of...          0


# 7. Export thành format SFT(Supervised Fine-Tuning)
- Mục đích: AI học được cách tuân thủ mệnh lệnh (instruction), lấy dữ liệu từ phần (input) để xử lý và sinh ra kết quả chuẩn xác ở phần (output).
- {
  "instruction": "Hãy dịch câu sau sang tiếng Anh.",
  "input": "Hôm nay trời mưa rất to.",
  "output": "It is raining very hard today."
}

In [18]:
import json

sft = []

for _, row in df.iterrows():

    item = {
        "instruction": "Analyze setiment of the review",
        "input": row["review"],
        "output": row["sentiment"]
    }
    sft.append(item)

In [19]:
# export
# .dumps chuyển đối tượng thành chuỗi json
with open("setiment.json", "w") as f:
    for item in sft:
        f.write(json.dumps(item) + "\n")
